In [1]:
from typing import Any, Callable
from sqlmodel import create_engine, select, Session
from sqlalchemy.engine import Engine
from sqlalchemy import event
from experiment import Model, Result, Celltype, Dataset, Sample, NumericArray
import pandas as pd
import mlflow
import logging
import polars as pl

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

/Users/egerc/Documents/Projects/notebook_repository/notebooks/2026-02-27T09-05-08Z/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
exp_name = "Default"
benchmark_experiment = mlflow.get_experiment_by_name(exp_name)
if not benchmark_experiment:
    raise ValueError(f"Experiment '{exp_name}' not found.")
runs = mlflow.search_runs(experiment_ids=[benchmark_experiment.experiment_id])
if runs.empty:
    raise RuntimeError(f"No runs found in experiment '{exp_name}'.")
last_run_id = runs.sort_values("start_time", ascending=False)["run_id"].iloc[0]
logger.info(f"Using run_id: {last_run_id}")
logger.info("Downloading 'database.db'...")
database_path = mlflow.artifacts.download_artifacts(
    run_id=last_run_id, artifact_path="database.db"
)
logger.info(f"Local path: {database_path}")
sqlite_url = f"sqlite:///{database_path}"
engine = create_engine(sqlite_url, echo=True)
logger.info("Database engine initialized.")

INFO: Using run_id: e112ee433c2648e69beb412944c74bf8
INFO: Downloading 'database.db'...
INFO: Local path: /var/folders/qm/v_v5_1r52bx792m7x2mh177c0000gn/T/tmp_p4_5cqp/database.db
INFO: Database engine initialized.


In [38]:
statment = select(Dataset, Celltype, Model, Result)

In [42]:
with Session(engine) as session:
    results = session.exec(statement).all()
results[0][2]

2026-03-04 16:46:12,353 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026/03/04 16:46:12 INFO sqlalchemy.engine.Engine: BEGIN (implicit)


2026-03-04 16:46:12,354 INFO sqlalchemy.engine.Engine SELECT dataset.id, dataset.name, celltype.id AS id_1, celltype.name AS name_1, celltype.counts_matrix, celltype.pca_embedding, celltype.umap_embedding, celltype.adjacency_matrix, celltype.dataset_id, result.id AS id_2, result.global_model_embedding, result.global_model_counts, result.celltype_model_embedding, result.celltype_model_counts, result.celltype_id, result.model_id, result.sample_id, model.id AS id_3, model.name AS name_2, sample.id AS id_4, sample.id_of_sample, sample.train_idx, sample.test_idx, sample.dataset_id AS dataset_id_1 
FROM dataset, celltype, result, model, sample 
WHERE model.name = ? AND result.sample_id = ?


2026/03/04 16:46:12 INFO sqlalchemy.engine.Engine: SELECT dataset.id, dataset.name, celltype.id AS id_1, celltype.name AS name_1, celltype.counts_matrix, celltype.pca_embedding, celltype.umap_embedding, celltype.adjacency_matrix, celltype.dataset_id, result.id AS id_2, result.global_model_embedding, result.global_model_counts, result.celltype_model_embedding, result.celltype_model_counts, result.celltype_id, result.model_id, result.sample_id, model.id AS id_3, model.name AS name_2, sample.id AS id_4, sample.id_of_sample, sample.train_idx, sample.test_idx, sample.dataset_id AS dataset_id_1 
FROM dataset, celltype, result, model, sample 
WHERE model.name = ? AND result.sample_id = ?


2026-03-04 16:46:12,355 INFO sqlalchemy.engine.Engine [cached since 214.6s ago] ('mock_predictor_1', 1)


2026/03/04 16:46:12 INFO sqlalchemy.engine.Engine: [cached since 214.6s ago] ('mock_predictor_1', 1)


2026-03-04 16:46:13,000 INFO sqlalchemy.engine.Engine ROLLBACK


2026/03/04 16:46:13 INFO sqlalchemy.engine.Engine: ROLLBACK


Result(global_model_embedding=array([[ 0.12573022, -0.13210486,  0.64042265,  0.10490012, -0.53566937],
       [ 0.36159505,  1.30400005,  0.94708096, -0.70373524, -1.26542147],
       [-0.62327446,  0.04132598, -2.32503077, -0.21879166, -1.24591095],
       ...,
       [ 0.50817789,  0.59103668,  0.26580883,  0.20518015,  0.930788  ],
       [-1.15304479, -1.24655195,  0.16245324,  0.05215079,  0.16366016],
       [ 1.59043298, -0.57930935,  0.75131876,  0.3844477 ,  2.3665624 ]],
      shape=(317, 5)), celltype_model_embedding=array([[ 0.12573022, -0.13210486,  0.64042265,  0.10490012, -0.53566937],
       [ 0.36159505,  1.30400005,  0.94708096, -0.70373524, -1.26542147],
       [-0.62327446,  0.04132598, -2.32503077, -0.21879166, -1.24591095],
       ...,
       [ 0.50817789,  0.59103668,  0.26580883,  0.20518015,  0.930788  ],
       [-1.15304479, -1.24655195,  0.16245324,  0.05215079,  0.16366016],
       [ 1.59043298, -0.57930935,  0.75131876,  0.3844477 ,  2.3665624 ]],
      sh

In [36]:
results[0][-1]

Sample(id_of_sample=0, test_idx=array([221, 434, 109, 334, 375,  71, 378, 205, 290, 181]), train_idx=array([182, 227, 371,  41,  55, 408, 142, 148, 292, 145, 235, 456,  15,
       209, 251, 418, 119, 411,  63, 440, 391,  54, 478, 269, 254, 220,
         5, 297, 212, 255, 374,   2, 345, 316, 340, 214, 324, 244,  94,
       348, 356, 111, 206, 277, 381, 483, 402, 453, 246,  98, 441, 242,
       435, 185,  88, 436, 126,  75, 321,  27, 335,  18,  39, 474, 448,
       379, 404, 485,  90,  84, 195, 201, 252, 386, 437, 491,  83, 336,
       133, 310, 162, 476,  89,  85, 155,  19, 477, 250, 232, 346, 353,
        99,  38, 203, 305, 196, 266, 103, 291, 486, 222, 147, 407, 167,
        36, 264, 136, 225, 202, 382, 445, 124, 164, 308, 471, 161,  52,
       274, 365, 160,  93, 355, 362, 383, 415,  44, 236, 354, 223, 403,
        77,  31,  32, 256, 380, 137, 329, 405, 300, 454, 475, 253, 153,
       420, 166, 361, 372, 200, 281, 468, 208,   0, 299,  43, 189,  46,
       229, 159, 128, 367, 198, 492

In [ ]:
MetricResult = list[dict[str, Any]]
Entry = tuple[Dataset, Celltype, Model, Sample, Result]
MetricResultFn = Callable[[Entry], MetricResult]


def construct_result_table(
    results: list[Entry],
    extract_results_fn: MetricResultFn,
) -> pl.DataFrame:

    rows: MetricResult = []

    for dataset, celltype, model, sample, result in results:
        extracted_results = extract_results_fn(
            (dataset, celltype, model, sample, result)
        )

        for extracted_result in extracted_results:
            rows.append(
                {
                    **extracted_result,
                    "dataset_name": dataset.name,
                    "celltype": celltype.name,
                    "sample_id": sample.id_of_sample,
                    "model_name": model.name,
                }
            )

    return pl.DataFrame(rows)